In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 1.3 (Rev 8 - 2x2 Layout + ROI Zoom + Thinner Lines)
----------------------------------------------------------
Updates vs Rev7:
1. FIXED: Reduced line width for 'Mixed' signal so it doesn't obscure comparison.
2. NEW: Added a 2x2 layout version.
   - Panel D is now a "Spectral Zoom" focusing on the region with the MAX difference between A and B.
   - Panel C includes a highlight span showing where Panel D is zooming in.
3. OUTPUT: Generates TWO files (Original layout fixed, and new 2x2 layout).

Inputs:
- Minimal .mat dataset (HDF5)
- Freesurfer LUT .xlsx/.csv
"""

import os
import re
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

# ================= Configuration =================

DATA_PATH = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat"
LUT_CSV_PATH = "/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx"

# --- Freeze region ---
USE_FIXED_BLOCK = True
FIXED_SLICE_IDX = 222
FIXED_BLOCK_TL = (36, 111)  # (row, col)

MANUAL_GRID_CENTER = None
GRID_RATIO = 3
IGNORE_LABELS = [0, 28, 60]

# Feature indexing
IDX_MPRAGE = 341
N_FEATURES = 341

# Output Filenames
OUTPUT_FILE_ORIG = "Figure_1_3_v7_Original_ThinLine.png"
OUTPUT_FILE_2x2 = "Figure_1_3_v7_2x2_Zoomed.png"

# Plotting Options
PLOT_SPECTRA_RAW = False
SPECTRA_PCTL_LOW = 1
SPECTRA_PCTL_HIGH = 99
CROP_MARGIN = 25

# Pretty display name options
STRIP_HEMISPHERE = True
STRIP_PREFIX_WM_CTX = False

# Zoom Window for Panel D (number of indices +/- around max diff)
ZOOM_WINDOW_RADIUS = 20

# ================= Utilities (Same as before) =================

def load_lut(file_path: str) -> dict:
    print(f"Loading LUT: {file_path} ...")
    try:
        if file_path.endswith((".xlsx", ".xls")):
            df = pd.read_excel(file_path)
        else:
            df = pd.read_csv(file_path, sep=None, engine="python")

        def find_col(candidates):
            for c in candidates:
                if c in df.columns:
                    return c
            return None

        col_id = find_col(["one_hot_loc_alex_label", "freesurfer_label", "idx"])
        col_name = find_col(["freesurfer_tissue_name", "tissue_name", "col label"])

        if (col_id is None) or (col_name is None):
            return {}

        valid = pd.to_numeric(df[col_id], errors="coerce").notna()
        ids = df.loc[valid, col_id].astype(int).values
        names = df.loc[valid, col_name].astype(str).values
        return dict(zip(ids, names))
    except Exception as e:
        print(f"[Warn] Failed to load LUT: {e}")
        return {}

def load_data_and_labels(mat_path: str):
    print(f"Loading Data: {mat_path} ...")
    with h5py.File(mat_path, "r") as f:
        data = f["data"][:]
        if data.shape[0] in [341, 351]:
            data = np.moveaxis(data, 0, -1)
        
        if "region_labels" in f:
            label_map = f["region_labels"][:]
        elif "one_hot_loc_alex_label" in f:
            one_hot = f["one_hot_loc_alex_label"][:]
            if one_hot.ndim == 4:
                axis = 0 if one_hot.shape[0] == 102 else -1
                label_map = np.argmax(one_hot, axis=axis)
            else:
                label_map = one_hot.astype(int)
        else:
            raise ValueError("Dataset missing labels!")
    return data, label_map.astype(int)

def robust_scale_01(vec, p_low=1, p_high=99):
    v = np.asarray(vec, dtype=np.float64)
    lo, hi = np.percentile(v, [p_low, p_high])
    den = hi - lo
    if den < 1e-12: den = 1.0
    out = (v - lo) / den
    return np.clip(out, 0.0, 1.0)

def pretty_label_name(name: str, strip_hemi=True, strip_prefix=False) -> str:
    if name is None: return ""
    s = str(name)
    if strip_prefix: s = re.sub(r"^(wm|ctx)-", "", s)
    if strip_hemi: s = s.replace("-rh-", "-").replace("-lh-", "-")
    return s.replace("_", "-")

def pick_top2_labels_by_frequency(block: np.ndarray, ignore_labels) -> tuple | None:
    vals, counts = np.unique(block, return_counts=True)
    mask = ~np.isin(vals, ignore_labels)
    vals, counts = vals[mask], counts[mask]
    if vals.size < 2: return None
    order = np.argsort(counts)[::-1]
    return int(vals[order[0]]), int(vals[order[1]]), int(min(counts[order[0]], counts[order[1]]))

def find_best_mixed_block(label_slice: np.ndarray, grid_ratio: int, lut: dict, ignore_labels):
    h, w = label_slice.shape
    best_score, best_pos, best_labels = -1, (h // 2, w // 2), (0, 0)
    for r in range(0, h - grid_ratio + 1, grid_ratio):
        for c in range(0, w - grid_ratio + 1, grid_ratio):
            block = label_slice[r:r + grid_ratio, c:c + grid_ratio]
            top2 = pick_top2_labels_by_frequency(block, ignore_labels)
            if top2:
                u1, u2, score = top2
                # heuristic preference
                n1, n2 = lut.get(u1, "").lower(), lut.get(u2, "").lower()
                if "wm" in n1 or "white" in n1: score += 1
                if "ctx" in n2 or "cortex" in n2: score += 1
                if score > best_score:
                    best_score, best_pos, best_labels = score, (r, c), (u1, u2)
    return best_pos, best_labels

def get_block_data(data, label_map, lut, slice_idx):
    """Helper to extract signals and crop images for plotting."""
    # ... (Logic extracted from original plot_rev7 for reuse) ...
    img_slice = data[slice_idx, :, :, IDX_MPRAGE]
    lbl_slice = label_map[slice_idx, :, :]
    
    # Norm image
    p1, p99 = np.percentile(img_slice, [1, 99])
    den = p99 - p1 if (p99-p1) > 1e-8 else 1.0
    img_disp = np.clip((img_slice - p1) / den, 0, 1)

    # Determine Block
    if USE_FIXED_BLOCK:
        r_b, c_b = int(FIXED_BLOCK_TL[0]), int(FIXED_BLOCK_TL[1])
        patch = lbl_slice[r_b:r_b+GRID_RATIO, c_b:c_b+GRID_RATIO]
        top2 = pick_top2_labels_by_frequency(patch, IGNORE_LABELS)
        if not top2: raise ValueError("Fixed block invalid.")
        target_labels = (top2[0], top2[1])
    elif MANUAL_GRID_CENTER:
        r_b = (MANUAL_GRID_CENTER[0] // GRID_RATIO) * GRID_RATIO
        c_b = (MANUAL_GRID_CENTER[1] // GRID_RATIO) * GRID_RATIO
        patch = lbl_slice[r_b:r_b+GRID_RATIO, c_b:c_b+GRID_RATIO]
        valid = [u for u in np.unique(patch) if u not in IGNORE_LABELS]
        target_labels = (valid[0], valid[1] if len(valid)>1 else valid[0])
    else:
        (r_b, c_b), target_labels = find_best_mixed_block(lbl_slice, GRID_RATIO, lut, IGNORE_LABELS)

    # Crop
    r_s, r_e = max(0, r_b - CROP_MARGIN), min(img_slice.shape[0], r_b + GRID_RATIO + CROP_MARGIN)
    c_s, c_e = max(0, c_b - CROP_MARGIN), min(img_slice.shape[1], c_b + GRID_RATIO + CROP_MARGIN)
    img_crop = img_disp[r_s:r_e, c_s:c_e]
    lbl_crop = lbl_slice[r_s:r_e, c_s:c_e]

    # Signals
    coords_A = []
    coords_B = []
    for i in range(GRID_RATIO):
        for j in range(GRID_RATIO):
            rr, cc = r_b + i, c_b + j
            val = int(lbl_slice[rr, cc])
            if val == target_labels[0]: coords_A.append((rr, cc))
            elif val == target_labels[1]: coords_B.append((rr, cc))

    vecs_A = [data[slice_idx, r, c, :N_FEATURES] for (r, c) in coords_A]
    vecs_B = [data[slice_idx, r, c, :N_FEATURES] for (r, c) in coords_B]
    vec_mixed = np.mean(data[slice_idx, r_b:r_b+GRID_RATIO, c_b:c_b+GRID_RATIO, :N_FEATURES], axis=(0, 1))

    sig_A = np.mean(vecs_A, axis=0) if vecs_A else np.zeros(N_FEATURES)
    sig_B = np.mean(vecs_B, axis=0) if vecs_B else np.zeros(N_FEATURES)
    
    if not PLOT_SPECTRA_RAW:
        sig_A = robust_scale_01(sig_A, SPECTRA_PCTL_LOW, SPECTRA_PCTL_HIGH)
        sig_B = robust_scale_01(sig_B, SPECTRA_PCTL_LOW, SPECTRA_PCTL_HIGH)
        sig_M = robust_scale_01(vec_mixed, SPECTRA_PCTL_LOW, SPECTRA_PCTL_HIGH)
    else:
        sig_M = vec_mixed

    # Metadata
    raw_nA = lut.get(int(target_labels[0]), f"ID {target_labels[0]}")
    raw_nB = lut.get(int(target_labels[1]), f"ID {target_labels[1]}")
    nA = pretty_label_name(raw_nA, STRIP_HEMISPHERE, STRIP_PREFIX_WM_CTX)
    nB = pretty_label_name(raw_nB, STRIP_HEMISPHERE, STRIP_PREFIX_WM_CTX)

    info = {
        "img_disp": img_disp, "img_crop": img_crop, "lbl_crop": lbl_crop,
        "sig_A": sig_A, "sig_B": sig_B, "sig_M": sig_M,
        "name_A": nA, "name_B": nB, "labels": target_labels,
        "bbox": (r_b, c_b), "crop_extent": [c_s, c_e, r_e, r_s],
        "crop_coords": (r_s, r_e, c_s, c_e)
    }
    return info

# ================= Plotting Function 1: Original Layout (Updated) =================

def plot_original_layout(info, output_file):
    """
    Original 1x3 layout.
    Fixes: Mixed line width reduced from 2 to 1.2.
    """
    fig = plt.figure(figsize=(18, 6), facecolor="white", constrained_layout=True)
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1.2, 1.5], wspace=0.25)

    # --- Panel A ---
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(info["img_disp"], cmap="gray", origin="upper")
    r_b, c_b = info["bbox"]
    c_s, c_e, r_e, r_s = info["crop_extent"] # Note: extent is [left, right, bottom, top] for matplotlib
    # Correction: info["crop_extent"] passed was actually for imshow extent logic,
    # let's use info["crop_coords"] to draw the green box
    rs, re, cs, ce = info["crop_coords"]
    
    ax1.add_patch(patches.Rectangle((cs, rs), ce-cs, re-rs, lw=1.5, edgecolor="lime", facecolor="none"))
    ax1.add_patch(patches.Rectangle((c_b, r_b), GRID_RATIO, GRID_RATIO, lw=2, edgecolor="red", facecolor="none"))
    ax1.set_title("(A) Full Slice", fontweight="bold")
    ax1.axis("off")

    # --- Panel B ---
    ax2 = fig.add_subplot(gs[1])
    ax2.imshow(info["img_crop"], cmap="gray", origin="upper", extent=info["crop_extent"])
    
    overlay = np.zeros((info["lbl_crop"].shape[0], info["lbl_crop"].shape[1], 4), dtype=np.float32)
    overlay[info["lbl_crop"] == info["labels"][0]] = [0, 0, 1, 0.40] # Blue
    overlay[info["lbl_crop"] == info["labels"][1]] = [0, 1, 0, 0.40] # Green
    ax2.imshow(overlay, origin="upper", extent=info["crop_extent"])

    # Grid lines
    grid_sc = (cs // GRID_RATIO) * GRID_RATIO
    grid_sr = (rs // GRID_RATIO) * GRID_RATIO
    for c in range(grid_sc, ce + 1, GRID_RATIO): ax2.axvline(c - 0.5, lw=0.5, alpha=0.5, color="cyan")
    for r in range(grid_sr, re + 1, GRID_RATIO): ax2.axhline(r - 0.5, lw=0.5, alpha=0.5, color="cyan")
    
    ax2.add_patch(patches.Rectangle((c_b - 0.5, r_b - 0.5), GRID_RATIO, GRID_RATIO, lw=2.5, edgecolor="red", facecolor="none"))
    ax2.set_title("(B) Mixed Block", fontweight="bold")
    ax2.axis("off")
    
    # Legend
    legs = [Line2D([0],[0], color="blue", lw=4, alpha=0.5), Line2D([0],[0], color="green", lw=4, alpha=0.5)]
    ax2.legend(legs, [info["name_A"], info["name_B"]], loc="lower center", bbox_to_anchor=(0.5, -0.15), fontsize=9)

    # --- Panel C ---
    ax3 = fig.add_subplot(gs[2])
    x = np.arange(N_FEATURES)
    ax3.plot(x, info["sig_A"], "-", alpha=0.6, lw=1, color="tab:blue", label=f"Pure '{info['name_A']}'")
    ax3.plot(x, info["sig_B"], "-", alpha=0.6, lw=1, color="tab:green", label=f"Pure '{info['name_B']}'")
    # FIX: Thinner mixed line
    ax3.plot(x, info["sig_M"], "-", alpha=0.9, lw=1.2, color="tab:red", label="Mixed (block mean)")

    ax3.set_title("(C) Spectral Mixing", fontweight="bold")
    ax3.set_xlabel("Feature Index")
    ax3.set_ylim(-0.05, 1.05) if not PLOT_SPECTRA_RAW else None
    ax3.legend(loc="upper right", fontsize=8)

    plt.savefig(output_file, dpi=300)
    print(f"Generated Original Layout: {output_file}")
    plt.close()

# ================= Plotting Function 2: 2x2 Layout with Zoom =================

def plot_2x2_layout(info, output_file):
    """
    New 2x2 Layout.
    Panel D is a zoom of the region with MAX difference between A and B.
    """
    fig = plt.figure(figsize=(14, 10), facecolor="white", constrained_layout=True)
    gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1], width_ratios=[1, 1])

    # --- Calculate Max Difference Region ---
    diff = np.abs(info["sig_A"] - info["sig_B"])
    idx_max = np.argmax(diff)
    zoom_start = max(0, idx_max - ZOOM_WINDOW_RADIUS)
    zoom_end = min(N_FEATURES, idx_max + ZOOM_WINDOW_RADIUS)
    
    # --- Panel A (Top Left) ---
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(info["img_disp"], cmap="gray", origin="upper")
    rs, re, cs, ce = info["crop_coords"]
    r_b, c_b = info["bbox"]
    ax1.add_patch(patches.Rectangle((cs, rs), ce-cs, re-rs, lw=1.5, edgecolor="lime", facecolor="none"))
    ax1.add_patch(patches.Rectangle((c_b, r_b), GRID_RATIO, GRID_RATIO, lw=2, edgecolor="red", facecolor="none"))
    ax1.set_title("(A) Full Slice", fontweight="bold")
    ax1.axis("off")

    # --- Panel B (Top Right) ---
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(info["img_crop"], cmap="gray", origin="upper", extent=info["crop_extent"])
    overlay = np.zeros((info["lbl_crop"].shape[0], info["lbl_crop"].shape[1], 4), dtype=np.float32)
    overlay[info["lbl_crop"] == info["labels"][0]] = [0, 0, 1, 0.40]
    overlay[info["lbl_crop"] == info["labels"][1]] = [0, 1, 0, 0.40]
    ax2.imshow(overlay, origin="upper", extent=info["crop_extent"])
    
    # Grid
    grid_sc = (cs // GRID_RATIO) * GRID_RATIO
    grid_sr = (rs // GRID_RATIO) * GRID_RATIO
    for c in range(grid_sc, ce + 1, GRID_RATIO): ax2.axvline(c - 0.5, lw=0.5, alpha=0.5, color="cyan")
    for r in range(grid_sr, re + 1, GRID_RATIO): ax2.axhline(r - 0.5, lw=0.5, alpha=0.5, color="cyan")
    ax2.add_patch(patches.Rectangle((c_b - 0.5, r_b - 0.5), GRID_RATIO, GRID_RATIO, lw=2.5, edgecolor="red", facecolor="none"))
    
    legs = [Line2D([0],[0], color="blue", lw=4, alpha=0.5), Line2D([0],[0], color="green", lw=4, alpha=0.5)]
    ax2.legend(legs, [info["name_A"], info["name_B"]], loc="lower right", fontsize=8)
    ax2.set_title("(B) Mixed Block Zoom", fontweight="bold")
    ax2.axis("off")

    # --- Panel C (Bottom Left) - Full Spectrum ---
    ax3 = fig.add_subplot(gs[1, 0])
    x = np.arange(N_FEATURES)
    ax3.plot(x, info["sig_A"], "-", alpha=0.6, lw=1, color="tab:blue", label=f"Pure A")
    ax3.plot(x, info["sig_B"], "-", alpha=0.6, lw=1, color="tab:green", label=f"Pure B")
    ax3.plot(x, info["sig_M"], "-", alpha=0.9, lw=1.2, color="tab:red", label="Mixed")
    
    # Highlight the zoom region
    ax3.axvspan(zoom_start, zoom_end, color='gold', alpha=0.2, label='Region in (D)')
    
    ax3.set_title("(C) Full Spectrum", fontweight="bold")
    ax3.set_xlabel("Feature Index")
    ax3.set_ylim(-0.05, 1.05) if not PLOT_SPECTRA_RAW else None
    ax3.legend(loc="upper right", fontsize=8)

    # --- Panel D (Bottom Right) - Zoomed Spectrum ---
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.plot(x, info["sig_A"], "-", alpha=0.6, lw=1.5, color="tab:blue", label=f"{info['name_A']}")
    ax4.plot(x, info["sig_B"], "-", alpha=0.6, lw=1.5, color="tab:green", label=f"{info['name_B']}")
    ax4.plot(x, info["sig_M"], "-", alpha=0.9, lw=2.0, color="tab:red", label="Mixed") # Slightly thicker here for clarity

    ax4.set_xlim(zoom_start, zoom_end)
    ax4.set_ylim(-0.05, 1.05) if not PLOT_SPECTRA_RAW else None
    
    # Highlight max difference point
    ax4.axvline(idx_max, color='black', linestyle='--', alpha=0.3)
    ax4.text(idx_max, 0.05, "Max Diff", rotation=90, verticalalignment='bottom', alpha=0.5, fontsize=8)
    
    # Background color to indicate zoom
    ax4.set_facecolor("#fffdf0") # Very light yellow

    ax4.set_title(f"(D) Zoom: Max Difference (Idx {idx_max})", fontweight="bold")
    ax4.set_xlabel("Feature Index")
    ax4.legend(loc="upper center", fontsize=8, ncol=1)

    plt.savefig(output_file, dpi=300)
    print(f"Generated 2x2 Layout: {output_file}")
    plt.close()

# ================= Main =================

if __name__ == "__main__":
    # 1. Load Data
    lut = load_lut(LUT_CSV_PATH)
    data, labels = load_data_and_labels(DATA_PATH)

    # 2. Determine Slice
    curr_slice = FIXED_SLICE_IDX if USE_FIXED_BLOCK else 160
    
    try:
        # 3. Process Data (Get signals, blocks, etc.)
        info_dict = get_block_data(data, labels, lut, curr_slice)
        
        # 4. Generate Plot 1: Original Layout (Thinner lines)
        plot_original_layout(info_dict, OUTPUT_FILE_ORIG)

        # 5. Generate Plot 2: 2x2 Layout (With Spectral Zoom)
        plot_2x2_layout(info_dict, OUTPUT_FILE_2x2)

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()